# MVPA Distinctiveness Analysis
Port of 02_strippeddown.ipynb to sym_pt project.

**Changes from long_pt version:**
1. Paths point to sym_pt processed directory
2. Subject info from unified long-format CSV via sym_pt_params
3. Brain mask from ses-01 applied before thresholding (new)

**Everything else matches long_pt:** same COPE_MAP, same 6mm sphere, same z>2.3 threshold, same distinctiveness metric.

In [9]:
# CELL 1: Setup & Configuration
import os, sys
import numpy as np
import nibabel as nib
import pandas as pd
import pickle
from pathlib import Path
from scipy.ndimage import label, center_of_mass
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import (processed_dir, skip_subs, is_patient,
                           get_sessions, get_sub_info, _load_csv)

BASE_DIR = Path(processed_dir)
OUTPUT_DIR = BASE_DIR / 'analyses' / 'mvpa_distinctiveness'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Same COPE_MAP as long_pt — used for BOTH ROI definition AND RSA
COPE_MAP = {'face': 1, 'house': 2, 'object': 3, 'word': 12}
BILATERAL_CATEGORIES = ['object', 'house']
UNILATERAL_CATEGORIES = ['face', 'word']

print(f'Base: {BASE_DIR}')
print(f'COPE_MAP: {COPE_MAP}')
print('Setup complete.')

Base: /user_data/csimmon2/sym_pt
COPE_MAP: {'face': 1, 'house': 2, 'object': 3, 'word': 12}
Setup complete.


In [10]:
# CELL 2: Load Subjects
df = _load_csv()

def load_subjects(patient_only=None):
    """Load subjects from unified CSV. patient_only: True/False/None(all)"""
    subjects = {}
    for sub_clean in sorted(df['sub_clean'].unique()):
        if sub_clean in skip_subs:
            continue
        sessions = get_sessions(sub_clean)
        if not sessions:
            continue
        info = get_sub_info(sub_clean, sessions[0])
        pt = is_patient(sub_clean)

        if patient_only is True and not pt:
            continue
        if patient_only is False and pt:
            continue

        # Check directory exists
        if not (BASE_DIR / f'sub-{sub_clean}').exists():
            continue

        intact = info.get('intact_hemi', '')
        if pt:
            hemi = 'l' if intact == 'left' else 'r'
        else:
            hemi = 'r'  # default for controls; left done separately

        subjects[f'sub-{sub_clean}'] = {
            'code': f"{info.get('group','')}{sub_clean}",
            'sessions': [f'{s:02d}' for s in sessions],
            'hemi': hemi,
            'group': info.get('group', 'unknown'),
            'patient_status': 'patient' if pt else 'control',
            'intact_hemi': intact
        }
    return subjects

ALL_PATIENTS = load_subjects(patient_only=True)
ALL_CONTROLS = load_subjects(patient_only=False)
ANALYSIS_SUBJECTS = {**ALL_PATIENTS, **ALL_CONTROLS}

print(f'Loaded {len(ANALYSIS_SUBJECTS)} subjects')
print(f'  Patients: {len(ALL_PATIENTS)}, Controls: {len(ALL_CONTROLS)}')

Loaded 50 subjects
  Patients: 26, Controls: 24


In [11]:
# CELL 3: ROI Extraction

def extract_rois(subject_id, subjects_dict, threshold_z=2.3):
    """Extract functional cluster ROIs across all sessions."""
    if subject_id not in subjects_dict:
        return {}

    info = subjects_dict[subject_id]
    code = info['code']
    hemi = info['hemi']
    sessions = info['sessions']
    first_session = sessions[0]

    print(f"{code} - Extracting ROIs [{info['group']} {info['patient_status']}, hemi={hemi}]")

    # Brain mask from FIRST session (all data registered to this space)
    brain_mask_file = BASE_DIR / subject_id / f'ses-{first_session}' / 'anat' / 'T1w_brain_mask.nii.gz'
    brain_mask = nib.load(str(brain_mask_file)).get_fdata() > 0 if brain_mask_file.exists() else None

    all_results = {}

    for category, cope_num in COPE_MAP.items():
        all_results[category] = {}

        # Search mask from first session
        mask_file = None
        for subdir in ['ROIs', os.path.join('derivatives', 'rois')]:
            p = BASE_DIR / subject_id / f'ses-{first_session}' / subdir / f'{hemi}_{category}_searchmask.nii.gz'
            if p.exists():
                mask_file = p
                break

        if mask_file is None:
            print(f"  {category}: mask not found")
            continue

        mask = nib.load(str(mask_file)).get_fdata() > 0
        affine = nib.load(str(mask_file)).affine

        for session in sessions:
            feat_dir = BASE_DIR / subject_id / f'ses-{session}' / 'derivatives' / 'fsl' / 'loc' / 'HighLevel.gfeat'

            zstat_name = 'zstat1.nii.gz' if session == first_session else f'zstat1_ses{first_session}.nii.gz'
            cope_file = feat_dir / f'cope{cope_num}.feat' / 'stats' / zstat_name

            if not cope_file.exists():
                continue

            zstat = nib.load(str(cope_file)).get_fdata()

            # Apply brain mask from first session
            if brain_mask is not None:
                zstat[~brain_mask] = 0

            suprathresh = (zstat > threshold_z) & mask

            if suprathresh.sum() < 50:
                continue

            labeled, n_clusters = label(suprathresh)
            if n_clusters == 0:
                continue

            cluster_sizes = [(labeled == i).sum() for i in range(1, n_clusters + 1)]
            largest_idx = np.argmax(cluster_sizes) + 1
            roi_mask = (labeled == largest_idx)

            peak_idx = np.unravel_index(np.argmax(zstat * roi_mask), zstat.shape)
            peak_z = zstat[peak_idx]
            centroid = nib.affines.apply_affine(affine, center_of_mass(roi_mask))

            all_results[category][session] = {
                'n_voxels': cluster_sizes[largest_idx - 1],
                'peak_z': peak_z,
                'centroid': centroid,
                'roi_mask': roi_mask
            }

    return all_results


print('\nEXTRACTING FUNCTIONAL ROIs')
print('=' * 70)

# Extract patient ROIs
functional_rois = {}
for subject_id in ALL_PATIENTS.keys():
    try:
        functional_rois[subject_id] = extract_rois(subject_id, ANALYSIS_SUBJECTS)
    except Exception as e:
        print(f"  {subject_id} failed: {e}")
        functional_rois[subject_id] = {}

# Extract control ROIs - RIGHT hemisphere
for subject_id in ALL_CONTROLS.keys():
    try:
        functional_rois[subject_id] = extract_rois(subject_id, ANALYSIS_SUBJECTS)
    except Exception as e:
        print(f"  {subject_id} failed: {e}")
        functional_rois[subject_id] = {}

print(f'\nExtracted {len(functional_rois)} subjects')

# Extract control ROIs - LEFT hemisphere
print('\nEXTRACTING CONTROLS LEFT HEMISPHERE')
print('=' * 70)

controls_left_functional = {}
for subject_id in ALL_CONTROLS.keys():
    temp_subjects = {subject_id: {**ANALYSIS_SUBJECTS[subject_id], 'hemi': 'l'}}
    try:
        controls_left_functional[subject_id] = extract_rois(subject_id, temp_subjects)
    except Exception as e:
        print(f"  {subject_id} failed: {e}")
        controls_left_functional[subject_id] = {}

print(f'\nExtracted left hemisphere for {len(controls_left_functional)} controls')


EXTRACTING FUNCTIONAL ROIs
patient004 - Extracting ROIs [patient patient, hemi=l]
patient007 - Extracting ROIs [patient patient, hemi=r]
patient008 - Extracting ROIs [patient patient, hemi=l]
patient010 - Extracting ROIs [patient patient, hemi=r]
patient017 - Extracting ROIs [patient patient, hemi=r]
patient021 - Extracting ROIs [patient patient, hemi=r]
patient045 - Extracting ROIs [patient patient, hemi=r]
patient047 - Extracting ROIs [patient patient, hemi=l]
patient049 - Extracting ROIs [patient patient, hemi=l]
patient066 - Extracting ROIs [patient patient, hemi=r]
patient069 - Extracting ROIs [patient patient, hemi=r]
patient070 - Extracting ROIs [patient patient, hemi=r]
patient072 - Extracting ROIs [patient patient, hemi=l]
patient073 - Extracting ROIs [patient patient, hemi=l]
patient074 - Extracting ROIs [patient patient, hemi=l]
patient075 - Extracting ROIs [patient patient, hemi=l]
patient076 - Extracting ROIs [patient patient, hemi=l]
patient077 - Extracting ROIs [patient

In [12]:
# CELL 4: RSA Analysis

def create_sphere(peak_coord, affine, brain_shape, radius=6):
    """Create 6mm sphere around peak."""
    grid_coords = np.array(np.meshgrid(
        np.arange(brain_shape[0]),
        np.arange(brain_shape[1]),
        np.arange(brain_shape[2]),
        indexing='ij'
    )).reshape(3, -1).T

    grid_world = nib.affines.apply_affine(affine, grid_coords)
    distances = np.linalg.norm(grid_world - peak_coord, axis=1)

    mask_3d = np.zeros(brain_shape, dtype=bool)
    within = grid_coords[distances <= radius]
    for coord in within:
        mask_3d[coord[0], coord[1], coord[2]] = True
    return mask_3d


def extract_betas(subject_id, session, sphere_mask, category_copes):
    """Extract beta patterns from sphere."""
    info = ANALYSIS_SUBJECTS[subject_id]
    first_session = info['sessions'][0]

    feat_dir = BASE_DIR / subject_id / f'ses-{session}' / 'derivatives' / 'fsl' / 'loc' / 'HighLevel.gfeat'

    beta_patterns = []
    valid_categories = []

    for category, cope_num in category_copes.items():
        cope_name = 'cope1.nii.gz' if session == first_session else f'cope1_ses{first_session}.nii.gz'
        cope_file = feat_dir / f'cope{cope_num}.feat' / 'stats' / cope_name

        if not cope_file.exists():
            continue

        cope_data = nib.load(str(cope_file)).get_fdata()
        roi_betas = cope_data[sphere_mask]
        roi_betas = roi_betas[np.isfinite(roi_betas)]

        if len(roi_betas) > 0:
            beta_patterns.append(roi_betas)
            valid_categories.append(category)

    if len(beta_patterns) == 0:
        return None, None

    min_voxels = min(len(b) for b in beta_patterns)
    beta_patterns = [b[:min_voxels] for b in beta_patterns]
    beta_matrix = np.column_stack(beta_patterns)
    return beta_matrix, valid_categories


def compute_rdm(beta_matrix, fisher_transform=True):
    """Compute RDM from beta patterns."""
    correlation_matrix = np.corrcoef(beta_matrix.T)
    rdm = 1 - correlation_matrix
    if fisher_transform:
        corr_fisher = np.arctanh(np.clip(correlation_matrix, -0.999, 0.999))
        return rdm, corr_fisher
    return rdm, correlation_matrix


def extract_rdms(functional_results, analysis_subjects):
    """Extract all RDMs from 6mm spheres."""
    all_rdms = {}

    for subject_id in analysis_subjects.keys():
        if subject_id not in functional_results:
            continue

        info = analysis_subjects[subject_id]
        code = info['code']
        sessions = info['sessions']
        first_session = sessions[0]

        ref_file = None
        for subdir in ['ROIs', os.path.join('derivatives', 'rois')]:
            p = BASE_DIR / subject_id / f'ses-{first_session}' / subdir / f"{info['hemi']}_face_searchmask.nii.gz"
            if p.exists():
                ref_file = p
                break
        if ref_file is None:
            continue

        ref_img = nib.load(str(ref_file))
        affine = ref_img.affine
        brain_shape = ref_img.shape

        print(f"{code}: RSA Analysis")
        all_rdms[subject_id] = {}

        for roi_name in COPE_MAP.keys():
            if roi_name not in functional_results[subject_id]:
                continue

            all_rdms[subject_id][roi_name] = {
                'rdms': {}, 'correlation_matrices': {}, 'beta_patterns': {},
                'valid_categories': None, 'session_peaks': {}, 'session_n_voxels': {}
            }

            for session in sessions:
                if session not in functional_results[subject_id][roi_name]:
                    continue

                peak = functional_results[subject_id][roi_name][session]['centroid']
                sphere_mask = create_sphere(peak, affine, brain_shape, radius=6)
                n_voxels = sphere_mask.sum()

                all_rdms[subject_id][roi_name]['session_peaks'][session] = peak
                all_rdms[subject_id][roi_name]['session_n_voxels'][session] = n_voxels

                beta_matrix, valid_cats = extract_betas(subject_id, session, sphere_mask, COPE_MAP)
                if beta_matrix is None:
                    continue

                rdm, corr_fisher = compute_rdm(beta_matrix, fisher_transform=True)

                all_rdms[subject_id][roi_name]['rdms'][session] = rdm
                all_rdms[subject_id][roi_name]['correlation_matrices'][session] = corr_fisher
                all_rdms[subject_id][roi_name]['beta_patterns'][session] = beta_matrix
                all_rdms[subject_id][roi_name]['valid_categories'] = valid_cats

    return all_rdms


def compute_liu_metrics(all_rdms, analysis_subjects):
    """Compute Liu's distinctiveness."""
    distinctiveness_results = {}
    roi_preferred = {'face': 'face', 'word': 'word', 'object': 'object', 'house': 'house'}

    for subject_id, categories in all_rdms.items():
        if subject_id not in analysis_subjects:
            continue

        distinctiveness_results[subject_id] = {}

        for roi_name, roi_data in categories.items():
            if not roi_data['correlation_matrices']:
                continue

            valid_cats = roi_data['valid_categories']
            if valid_cats is None or len(valid_cats) < 4:
                continue

            preferred_cat = roi_preferred[roi_name]
            if preferred_cat not in valid_cats:
                continue

            pref_idx = valid_cats.index(preferred_cat)
            nonpref_indices = [i for i, cat in enumerate(valid_cats) if cat != preferred_cat]

            distinctiveness_results[subject_id][roi_name] = {}

            for session, corr_matrix in roi_data['correlation_matrices'].items():
                pref_vs_nonpref = corr_matrix[pref_idx, nonpref_indices]
                mean_corr = np.mean(pref_vs_nonpref)

                distinctiveness_results[subject_id][roi_name][session] = {
                    'liu_distinctiveness': mean_corr,
                    'individual_correlations': pref_vs_nonpref
                }

    return distinctiveness_results


print('\nEXTRACTING RSA DATA')
print('=' * 70)

all_rdms = extract_rdms(functional_rois, ANALYSIS_SUBJECTS)
liu_distinctiveness = compute_liu_metrics(all_rdms, ANALYSIS_SUBJECTS)

print('\nCONTROLS LEFT HEMISPHERE RSA')
print('=' * 70)
controls_left_rdms = extract_rdms(controls_left_functional, ALL_CONTROLS)
controls_left_distinctiveness = compute_liu_metrics(controls_left_rdms, ALL_CONTROLS)

print('\nRSA analysis complete!')


EXTRACTING RSA DATA
patient004: RSA Analysis


patient007: RSA Analysis
patient008: RSA Analysis
patient010: RSA Analysis
patient017: RSA Analysis
patient021: RSA Analysis
patient045: RSA Analysis
patient047: RSA Analysis
patient049: RSA Analysis
patient066: RSA Analysis
patient069: RSA Analysis
patient070: RSA Analysis
patient072: RSA Analysis
patient073: RSA Analysis
patient074: RSA Analysis
patient075: RSA Analysis
patient076: RSA Analysis
patient077: RSA Analysis
patient078: RSA Analysis
patient079: RSA Analysis
patient081: RSA Analysis
patient086: RSA Analysis
patient089: RSA Analysis
patient090: RSA Analysis
patient091: RSA Analysis
patient092: RSA Analysis
control018: RSA Analysis
control022: RSA Analysis
control025: RSA Analysis
control027: RSA Analysis
control038: RSA Analysis
control052: RSA Analysis
control057: RSA Analysis
control058: RSA Analysis
control059: RSA Analysis
control062: RSA Analysis
control064: RSA Analysis
control067: RSA Analysis
control068: RSA Analysis
control071: RSA Analysis
control083: RSA Analysis


In [17]:
# CELL 5: Spatial Analysis

def get_bootstrapped_error_radius(pair_peaks, n_bootstraps=1000):
    """Calculate bootstrapped measurement error radius."""
    if not pair_peaks or len(pair_peaks) < 2:
        return 1.0
    data = np.array([p['coord'][:2] for p in pair_peaks])
    def stat_func(coords):
        if len(np.unique(coords[:, 0])) < 2 or len(np.unique(coords[:, 1])) < 2:
            return 0.0
        return np.sqrt(np.std(coords[:, 0])**2 + np.std(coords[:, 1])**2)
    bootstrapped = [stat_func(data[np.random.choice(len(data), len(data), replace=True)])
                    for _ in range(n_bootstraps)]
    result = np.mean(bootstrapped)
    return result if not np.isnan(result) and result > 0 else stat_func(data)


def calc_error_radii(functional_results, analysis_subjects):
    radii = {}
    for subject_id in analysis_subjects.keys():
        if subject_id not in functional_results:
            continue
        radii[subject_id] = {}
        for category, sessions_data in functional_results[subject_id].items():
            if len(sessions_data) < 2:
                radii[subject_id][category] = 1.0
                continue
            pair_peaks = [{'coord': data['centroid'], 'session': session}
                         for session, data in sessions_data.items()]
            radii[subject_id][category] = get_bootstrapped_error_radius(pair_peaks)
    return radii


def calc_drift(functional_results, radii, analysis_subjects):
    drift_results = {}
    for subject_id, categories in functional_results.items():
        if subject_id not in analysis_subjects:
            continue
        drift_results[subject_id] = {}
        for category, sessions_data in categories.items():
            if len(sessions_data) < 2:
                continue
            sessions = sorted(sessions_data.keys())
            baseline_session = sessions[0]
            baseline_centroid = sessions_data[baseline_session]['centroid']
            error_radius = radii[subject_id].get(category, 1.0)
            drift_results[subject_id][category] = {
                'baseline_session': baseline_session,
                'baseline_centroid': baseline_centroid,
                'error_radius': error_radius,
                'from_baseline_drift': []
            }
            for session in sessions[1:]:
                current_centroid = sessions_data[session]['centroid']
                drift_distance = np.linalg.norm(current_centroid - baseline_centroid)
                drift_results[subject_id][category]['from_baseline_drift'].append({
                    'session': session,
                    'distance_mm': drift_distance,
                    'relative_to_error': drift_distance / error_radius
                })
    return drift_results


def calc_hemisphere_effects(drift_data, distinctiveness_data, analysis_subjects,
                           controls_left_drift=None, controls_left_distinct=None):
    """Calculate hemisphere-specific effects including controls both hemispheres."""
    table_data = []

    for subject_id in analysis_subjects.keys():
        info = analysis_subjects[subject_id]
        code = info['code']

        if info['patient_status'] == 'control' and controls_left_drift:
            for hemi_suffix, hemi_label, drift_source, distinct_source in [
                ('_R', 'r', drift_data.get(subject_id, {}), distinctiveness_data.get(subject_id, {})),
                ('_L', 'l', controls_left_drift.get(subject_id, {}),
                 controls_left_distinct.get(subject_id, {}) if controls_left_distinct else {})
            ]:
                spatial_data = {}
                repr_data = {}
                for category, drift_info in drift_source.items():
                    if drift_info.get('from_baseline_drift'):
                        spatial_data[category] = np.mean([d['distance_mm'] for d in drift_info['from_baseline_drift']])
                for category, sessions in distinct_source.items():
                    session_keys = sorted(sessions.keys())
                    if len(session_keys) >= 2:
                        baseline = sessions[session_keys[0]]['liu_distinctiveness']
                        final = sessions[session_keys[-1]]['liu_distinctiveness']
                        repr_data[category] = abs(final - baseline)
                for category in COPE_MAP.keys():
                    if category in spatial_data:
                        table_data.append({
                            'Subject': code + hemi_suffix,
                            'Group': info['group'],
                            'Status': info['patient_status'],
                            'Hemisphere': hemi_label,
                            'Category': category.title(),
                            'Category_Type': 'Bilateral' if category in BILATERAL_CATEGORIES else 'Unilateral',
                            'Spatial_Drift_mm': round(spatial_data[category], 2),
                            'Representational_Change': round(repr_data.get(category, 0), 3),
                            'Sessions': len(analysis_subjects[subject_id]['sessions'])
                        })
        else:
            spatial_data = {}
            repr_data = {}
            hemi_label = info['hemi']
            if subject_id in drift_data:
                for category, drift_info in drift_data[subject_id].items():
                    if drift_info.get('from_baseline_drift'):
                        spatial_data[category] = np.mean([d['distance_mm'] for d in drift_info['from_baseline_drift']])
            if subject_id in distinctiveness_data:
                for category, sessions in distinctiveness_data[subject_id].items():
                    session_keys = sorted(sessions.keys())
                    if len(session_keys) >= 2:
                        baseline = sessions[session_keys[0]]['liu_distinctiveness']
                        final = sessions[session_keys[-1]]['liu_distinctiveness']
                        repr_data[category] = abs(final - baseline)
            for category in COPE_MAP.keys():
                if category in spatial_data:
                    table_data.append({
                        'Subject': code,
                        'Group': info['group'],
                        'Status': info['patient_status'],
                        'Hemisphere': hemi_label,
                        'Category': category.title(),
                        'Category_Type': 'Bilateral' if category in BILATERAL_CATEGORIES else 'Unilateral',
                        'Spatial_Drift_mm': round(spatial_data[category], 2),
                        'Representational_Change': round(repr_data.get(category, 0), 3),
                        'Sessions': len(analysis_subjects[subject_id]['sessions'])
                    })

    return pd.DataFrame(table_data)


print('CALCULATING SPATIAL METRICS')
print('=' * 70)

error_radii = calc_error_radii(functional_rois, ANALYSIS_SUBJECTS)
drift_data = calc_drift(functional_rois, error_radii, ANALYSIS_SUBJECTS)

error_radii_left = calc_error_radii(controls_left_functional, ALL_CONTROLS)
controls_left_drift = calc_drift(controls_left_functional, error_radii_left, ALL_CONTROLS)

results_table = calc_hemisphere_effects(drift_data, liu_distinctiveness, ANALYSIS_SUBJECTS,
                                       controls_left_drift, controls_left_distinctiveness)

print(f'\nAnalysis complete: {len(results_table)} data points')

CALCULATING SPATIAL METRICS

Analysis complete: 0 data points


In [14]:
# CELL 6: Main Group Analysis

def analyze_groups(results_table):
    print('THREE-GROUP COMPARISON: OTC vs nonOTC vs Controls')
    print('=' * 70)

    clean_data = results_table[results_table['Category_Type'] != 'Summary'].copy()

    control_data = clean_data[clean_data['Status'] == 'control'].copy()
    control_data['Subject_Base'] = control_data['Subject'].str.replace('_L|_R', '', regex=True)
    control_averaged = control_data.groupby(['Subject_Base', 'Category', 'Category_Type']).agg({
        'Spatial_Drift_mm': 'mean',
        'Representational_Change': 'mean'
    }).reset_index()

    patient_data = clean_data[clean_data['Status'] == 'patient']
    otc = patient_data[patient_data['Group'] == 'OTC']
    nonotc = patient_data[patient_data['Group'] == 'nonOTC']
    controls = control_averaged

    print(f"\nREPRESENTATIONAL CHANGE:")
    print(f"{'Group':<15} {'Bilateral':<12} {'Unilateral':<12} {'Difference':<12}")
    print('-' * 52)

    for name, data in [('OTC', otc), ('nonOTC', nonotc), ('Controls', controls)]:
        if len(data) == 0:
            print(f'{name:<15} no data')
            continue
        bil = data[data['Category_Type'] == 'Bilateral']['Representational_Change'].mean()
        uni = data[data['Category_Type'] == 'Unilateral']['Representational_Change'].mean()
        diff = bil - uni
        print(f'{name:<15} {bil:<12.3f} {uni:<12.3f} {diff:<12.3f}')

    print(f'\n  Controls by hemisphere:')
    for hemi, hemi_label in [('l', 'Left'), ('r', 'Right')]:
        hemi_data = control_data[control_data['Hemisphere'] == hemi]
        if len(hemi_data) == 0:
            continue
        bil = hemi_data[hemi_data['Category_Type'] == 'Bilateral']['Representational_Change'].mean()
        uni = hemi_data[hemi_data['Category_Type'] == 'Unilateral']['Representational_Change'].mean()
        diff = bil - uni
        print(f"    {hemi_label:<6} {bil:<12.3f} {uni:<12.3f} {diff:<12.3f}")


print('MAIN GROUP ANALYSIS')
print('=' * 50)
if len(results_table) > 0:
    analyze_groups(results_table)
else:
    print('No results yet.')
print('\nAnalysis complete!')

MAIN GROUP ANALYSIS
No results yet.

Analysis complete!


In [15]:
# CELL 7: Verification

print('=' * 70)
print('VERIFICATION')
print('=' * 70)

print(f'\n1. BASIC COUNTS:')
print(f'   Total rows: {len(results_table)}')
if len(results_table) > 0:
    print(f'   Unique subjects: {results_table["Subject"].nunique()}')

print(f'\n2. CONTROL HEMISPHERES:')
if len(results_table) > 0:
    controls = results_table[results_table['Status'] == 'control']
    has_L = any('_L' in str(s) for s in controls['Subject'].unique())
    has_R = any('_R' in str(s) for s in controls['Subject'].unique())
    print(f'   Has _L: {has_L}, Has _R: {has_R}')

print(f'\n3. GROUP COUNTS:')
if len(results_table) > 0:
    for group in ['OTC', 'nonOTC']:
        n = len(results_table[results_table['Group'] == group])
        print(f'   {group}: {n} rows')
    n_ctrl = len(results_table[results_table['Status'] == 'control'])
    print(f'   Controls: {n_ctrl} rows')

print(f'\n4. PARAMETERS:')
print(f'   COPE_MAP: {COPE_MAP}')
print(f'   Sphere: 6mm, z>2.3, min 50 voxels')

VERIFICATION

1. BASIC COUNTS:
   Total rows: 0

2. CONTROL HEMISPHERES:

3. GROUP COUNTS:

4. PARAMETERS:
   COPE_MAP: {'face': 1, 'house': 2, 'object': 3, 'word': 12}
   Sphere: 6mm, z>2.3, min 50 voxels


In [16]:
# CELL 8: Save Results (cluster only, not git)

if len(results_table) > 0:
    results_table.to_csv(OUTPUT_DIR / 'results_table.csv', index=False)

results_to_save = {
    'functional_rois': functional_rois,
    'liu_distinctiveness': liu_distinctiveness,
    'drift_data': drift_data,
    'results_table': results_table,
    'controls_left_drift': controls_left_drift,
    'controls_left_distinctiveness': controls_left_distinctiveness
}

with open(OUTPUT_DIR / 'mvpa_results.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print(f'Saved to: {OUTPUT_DIR}')
print(f'  results_table.csv')
print(f'  mvpa_results.pkl')
print('\nDone!')

Saved to: /user_data/csimmon2/sym_pt/analyses/mvpa_distinctiveness
  results_table.csv
  mvpa_results.pkl

Done!
